## Install PySpark and create a session

In [1]:
!pip install pyspark==4.1.2

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import DataFrameReader

#create a spark session
spark=SparkSession.builder.appName("AirQuality").getOrCreate()
spark

## Read in data to pyspark dataframe

In [3]:
#Mount Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
#define folder path
path='/content/drive/MyDrive/AirQualityModel/SiteData'

#import datetime
from datetime import datetime,date,time

#Read a single CSV to check inferred schema
df=spark.read.csv('/content/drive/MyDrive/AirQualityModel/SiteData/AQ_Site1.csv',header=True,inferSchema=True)

In [5]:
#Check the top 5 rows to see that data has read in as expected
df.show(5)

+----------+--------+------------------------------------------+----------------+-----------------------+-------+-------------------+-------+--------------------+---------+---------+----------------+---------------+-------+
|      Date|    Time|PM2.5 particulate matter (Hourly measured)|         Status3|Modelled Wind Direction|Status5|Modelled Wind Speed|Status7|           Site Name| Latitude|Longitude|       Site Type|Local Authority|   Zone|
+----------+--------+------------------------------------------+----------------+-----------------------+-------+-------------------+-------+--------------------+---------+---------+----------------+---------------+-------+
|01/01/2025|01:00:00|                                     7.453|V ugm-3 (Ref.eq)|                  231.4|  N deg|               10.9| N ms-1|Borehamwood Meado...|51.661229| -0.27055|Urban Background|      Hertsmere|Eastern|
|01/01/2025|02:00:00|                                     4.151|V ugm-3 (Ref.eq)|                  231.9

In [6]:
#Check the schema
df.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- PM2.5 particulate matter (Hourly measured): string (nullable = true)
 |-- Status3: string (nullable = true)
 |-- Modelled Wind Direction: string (nullable = true)
 |-- Status5: string (nullable = true)
 |-- Modelled Wind Speed: string (nullable = true)
 |-- Status7: string (nullable = true)
 |-- Site Name: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Site Type: string (nullable = true)
 |-- Local Authority: string (nullable = true)
 |-- Zone: string (nullable = true)



#The inferred schema has mis-defined some fields as strings, when they should be dates, times floats and doubles

Define a new schema that better matches the expected data

In [21]:
from pyspark.sql.types import StructType, StructField, StringType, FloatType,DoubleType,TimestampType,DateType,TimeType

#define the schema
schema = StructType([
    StructField("Date", DateType(), nullable=True),
    StructField("Time", StringType(), nullable=True),
    StructField("Particulate reading", DoubleType(), nullable=True),
    StructField("Particulate Status", StringType(),nullable=True),
    StructField("Wind Direction",FloatType(), nullable=True),
    StructField("Wind Direction Status",StringType(),nullable=True),
    StructField("Wind Speed",FloatType(),nullable=True),
    StructField("wind Speed Status",StringType(),nullable=True),
    StructField("Site Name",StringType(),nullable=True),
    StructField("Latitude",DoubleType(),nullable=True),
    StructField("Longitude",DoubleType(),nullable=True),
    StructField("SiteType",StringType(),nullable=True),
    StructField("Local Authority",StringType(),nullable=True),
    StructField("Zone",StringType(),nullable=True)
    ])

#Read a single CSV to confirm that new schema has been applied
df2=spark.read.csv('/content/drive/MyDrive/AirQualityModel/SiteData/AQ_Site1.csv',header=True,schema=schema)
df2.printSchema()



root
 |-- Date: date (nullable = true)
 |-- Time: string (nullable = true)
 |-- Particulate reading: double (nullable = true)
 |-- Particulate Status: string (nullable = true)
 |-- Wind Direction: float (nullable = true)
 |-- Wind Direction Status: string (nullable = true)
 |-- Wind Speed: float (nullable = true)
 |-- wind Speed Status: string (nullable = true)
 |-- Site Name: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- SiteType: string (nullable = true)
 |-- Local Authority: string (nullable = true)
 |-- Zone: string (nullable = true)



In [22]:
df2.show(5)

+----+--------+-------------------+------------------+--------------+---------------------+----------+-----------------+--------------------+---------+---------+----------------+---------------+-------+
|Date|    Time|Particulate reading|Particulate Status|Wind Direction|Wind Direction Status|Wind Speed|wind Speed Status|           Site Name| Latitude|Longitude|        SiteType|Local Authority|   Zone|
+----+--------+-------------------+------------------+--------------+---------------------+----------+-----------------+--------------------+---------+---------+----------------+---------------+-------+
|NULL|01:00:00|              7.453|  V ugm-3 (Ref.eq)|         231.4|                N deg|      10.9|           N ms-1|Borehamwood Meado...|51.661229| -0.27055|Urban Background|      Hertsmere|Eastern|
|NULL|02:00:00|              4.151|  V ugm-3 (Ref.eq)|         231.9|                N deg|      11.3|           N ms-1|Borehamwood Meado...|51.661229| -0.27055|Urban Background|      Hert

# Having confirmed the new schema is being applied on a single CSV, we now read all the CSVs together into a single dataframe

However, we can see that the data has returned as NULL, as it is not in the default format - when loading in the full dataset, we will use the dataFormat option to define the date format used

In [23]:
#read all csv files in folder to single dataframe
df=(spark.read
    .format('csv')
    .option('header',True)
    .option('dateFormat','dd/MM/yyyy')
    .schema(schema)
    .load(path))
#Check the schema
df.printSchema()


root
 |-- Date: date (nullable = true)
 |-- Time: string (nullable = true)
 |-- Particulate reading: double (nullable = true)
 |-- Particulate Status: string (nullable = true)
 |-- Wind Direction: float (nullable = true)
 |-- Wind Direction Status: string (nullable = true)
 |-- Wind Speed: float (nullable = true)
 |-- wind Speed Status: string (nullable = true)
 |-- Site Name: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- SiteType: string (nullable = true)
 |-- Local Authority: string (nullable = true)
 |-- Zone: string (nullable = true)



# Initial Data overview and exploration

In [24]:
#First, lets look at the tope rows of the data
df.show(5)

+----------+--------+-------------------+------------------+--------------+---------------------+----------+-----------------+--------------------+---------+---------+----------------+---------------+-------+
|      Date|    Time|Particulate reading|Particulate Status|Wind Direction|Wind Direction Status|Wind Speed|wind Speed Status|           Site Name| Latitude|Longitude|        SiteType|Local Authority|   Zone|
+----------+--------+-------------------+------------------+--------------+---------------------+----------+-----------------+--------------------+---------+---------+----------------+---------------+-------+
|2025-01-01|01:00:00|             12.618|  V ugm-3 (Ref.eq)|         236.5|                N deg|      13.7|           N ms-1|Peterborough Gart...|52.594012|-0.248945|Urban Background|   Peterborough|Eastern|
|2025-01-01|02:00:00|              3.325|  V ugm-3 (Ref.eq)|         236.0|                N deg|      14.2|           N ms-1|Peterborough Gart...|52.594012|-0.2489

Time has been left as a string - this will need to be combined with the date into a single field and converted to a time type

In [25]:
#Start with using the describe function on the core columns (non-status) to get a picture of the data
df.select("Date","Particulate reading","Wind Direction","Wind Speed").describe().show()

+-------+-------------------+-----------------+------------------+
|summary|Particulate reading|   Wind Direction|        Wind Speed|
+-------+-------------------+-----------------+------------------+
|  count|              94254|            94512|             94512|
|   mean|  8.635067296878514| 190.751974356321| 4.924885728524972|
| stddev|  7.937105028580018|96.91355478251096|2.5305540503679733|
|    min|               -4.0|              0.0|               0.0|
|    max|            179.647|            360.0|              21.7|
+-------+-------------------+-----------------+------------------+



#Create additional fields to add context


# Create test and train datasets, ensuring that splits are suitably stratified

# Explore training dataset

# Explore models for predicting air quality levels

# Select the best model and use cross-validation to tune the hyperparameters

# Create pipeline to prepare data and run selected model on it

#Run pipeline on Test data